Practice set 1

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .getOrCreate()
products_data = [
    ("P01", "Laptop", "Electronics", 60000),
    ("P02", "Monitor", "Electronics", 14000),
    ("P03", "Headphones", "Accessories", 3500),
    ("P04", "Printer", "Office", 17000),
    ("P05", "Router", "Networking", 4500),
    ("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "list_price"]
)
stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
    stores_data,
    ["store_id", "city", "region"]
)
products_df.show()
stores_df.show()
sales_df = (
    spark.range(1, 100001)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "order_date",
        F.date_add(
            F.to_date(F.lit("2026-01-01")),
            (F.col("order_id") % 180).cast("int")
        )
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("P"),
            F.lpad(
                ((F.col("order_id") % 6) + 1).cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "store_id",
        F.concat(
            F.lit("S"),
            F.lpad(
                ((F.floor(F.col("order_id") / 3) % 4) + 1)
                .cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "quantity",
        ((F.col("order_id") % 8) + 1).cast("int")
    )
    .withColumn(
        "discount_pct",
        F.when(
            F.col("order_id") % 19 == 0,
            F.lit(None).cast("int")
        ).otherwise(
            ((F.floor(F.col("order_id") / 5) % 4) * 5)
            .cast("int")
        )
    )
    .withColumn(
        "order_status",
        F.when(F.col("order_id") % 11 == 0, "Cancelled")
         .when(F.col("order_id") % 7 == 0, "Returned")
         .otherwise("Completed")
    )

    .withColumn(
        "payment_method",
        F.when(F.col("order_id") % 3 == 0, "UPI")
         .when(F.col("order_id") % 3 == 1, "Card")
         .otherwise("Cash")
    )

    .withColumn(
        "salesperson_id",
        F.when(
            F.col("order_id") % 17 == 0,
            F.lit(None).cast("string")
        ).otherwise(
            F.concat(
                F.lit("EMP"),
                F.lpad(
                    ((F.col("order_id") % 12) + 1).cast("string"),
                    2, "0"
                )
            )
        )
    )

    .join(
        products_df.select(
            "product_id",
            F.col("list_price").alias("unit_price")
        ),
        "product_id",
        "left"
    )

    .select(
        "order_id", "order_date", "store_id",
        "product_id", "quantity", "unit_price",
        "discount_pct", "order_status",
        "payment_method", "salesperson_id"
    )
)

print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)

+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+

Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_p

In [ ]:
ans_1 = sales_df.select("order_id", "product_id", "quantity", "unit_price")
ans_1.show(5)

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       2|       P03|       3|      3500|
|       4|       P05|       5|      4500|
|       5|       P06|       6|     24000|
|       1|       P02|       2|     14000|
+--------+----------+--------+----------+
only showing top 5 rows


In [ ]:
ans_2 = sales_df.filter((F.col("order_status") == "Completed") & (F.col("quantity") > 5))
ans_2.show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|
|      23|2026-01-24|     S04|       P06|       8|     24000|           0|   Completed|          Cash|         EMP12|
|      29|2026-01-30|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|
|      13|2026-01-14|     S01|       P02|       6|     14000|          10|   Completed|          Card|         EMP02|
+--------+----------+--------+----------+--------+------

In [ ]:
ans_3 = sales_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
ans_3.show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|        68000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|        10500|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|   Completed|          Card|         EMP05|        22500|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|       144000|
|       1|2026-01-02|     S01|       P02|       2|     14000|           0|  

In [ ]:
ans_4 = sales_df.withColumnRenamed("order_status", "transaction_status").drop("payment_method")
ans_4.show(5)

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|         Completed|         EMP03|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|         Completed|         EMP05|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|         Completed|         EMP06|
|       1|2026-01-02|     S01|       P02|       2|     14000|           0|         Completed|         EMP02|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
only showing top 5 

In [ ]:
ans_5 = sales_df.groupBy("product_id").agg(F.sum("quantity").alias("total_quantity"))
ans_5.show()

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|       P04|         83334|
|       P03|         66667|
|       P05|         66665|
|       P06|         83330|
|       P02|         83336|
|       P01|         66668|
+----------+--------------+



In [ ]:
ans_6 = sales_df.join(products_df, "product_id", "left") \
                .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
                .groupBy("category") \
                .agg(F.sum("gross_revenue").alias("total_gross_revenue"))
ans_6.show()

+-----------+-------------------+
|   category|total_gross_revenue|
+-----------+-------------------+
|     Office|         1416678000|
| Networking|          299992500|
|Electronics|         7166704000|
|Accessories|          233334500|
+-----------+-------------------+



In [ ]:
ans_7 = sales_df.join(stores_df, "store_id", "left") \
                .groupBy("city") \
                .agg(F.count("order_id").alias("total_transactions"))
ans_7.show()

+---------+------------------+
|     city|total_transactions|
+---------+------------------+
|   Mumbai|             24999|
|    Delhi|             24999|
|Bengaluru|             25001|
|Hyderabad|             25001|
+---------+------------------+



In [ ]:
ans_8 = sales_df.filter(F.col("order_status") == "Completed") \
                .join(products_df, "product_id", "left") \
                .join(stores_df, "store_id", "left") \
                .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
                .groupBy("product_id", "product_name") \
                .agg(F.sum("gross_revenue").alias("total_gross_revenue")) \
                .orderBy(F.col("total_gross_revenue").desc()) \
                .limit(5)
ans_8.show()

+----------+------------+-------------------+
|product_id|product_name|total_gross_revenue|
+----------+------------+-------------------+
|       P01|      Laptop|         3116940000|
|       P06|      Tablet|         1558416000|
|       P04|     Printer|         1103912000|
|       P02|     Monitor|          909076000|
|       P05|      Router|          233766000|
+----------+------------+-------------------+



In [ ]:
sales_df.select(
    F.count(F.when(F.col("discount_pct").isNull(), 1)).alias("null_discount_count"),
    F.count(F.when(F.col("salesperson_id").isNull(), 1)).alias("null_salesperson_count")
).show()

+-------------------+----------------------+
|null_discount_count|null_salesperson_count|
+-------------------+----------------------+
|               5263|                  5882|
+-------------------+----------------------+



In [ ]:
ans_10 = sales_df.fillna(0, subset=["discount_pct"])
ans_10.show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|   Completed|          Card|         EMP05|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|
|       1|2026-01-02|     S01|       P02|       2|     14000|           0|   Completed|          Card|         EMP02|
+--------+----------+--------+----------+--------+------

In [ ]:
ans_11 = sales_df.dropna(subset=["salesperson_id"])
ans_11.show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|   Completed|          Card|         EMP05|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|
|       1|2026-01-02|     S01|       P02|       2|     14000|           0|   Completed|          Card|         EMP02|
+--------+----------+--------+----------+--------+------

In [ ]:
ans_12 = sales_df.withColumn(
    "order_size",
    F.when(F.col("quantity").between(1, 3), "Small")
     .when(F.col("quantity").between(4, 6), "Medium")
     .otherwise("Large")
)
ans_12.show(5)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+----------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|order_size|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+----------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|    Medium|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|     Small|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|   Completed|          Card|         EMP05|    Medium|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|    Medium|
|       1|2026-01-02|     S01|       P02|       2|     14000|           0|   Completed|          

In [ ]:
cat_product_rev = sales_df.join(products_df, "product_id", "left") \
                          .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
                          .groupBy("category", "product_id", "product_name") \
                          .agg(F.sum("gross_revenue").alias("total_gross_revenue"))

window_spec_cat = Window.partitionBy("category").orderBy(F.col("total_gross_revenue").desc())

ans_13 = cat_product_rev.withColumn("rank", F.dense_rank().over(window_spec_cat))
ans_13.show()


+-----------+----------+------------+-------------------+----+
|   category|product_id|product_name|total_gross_revenue|rank|
+-----------+----------+------------+-------------------+----+
|Accessories|       P03|  Headphones|          233334500|   1|
|Electronics|       P01|      Laptop|         4000080000|   1|
|Electronics|       P06|      Tablet|         1999920000|   2|
|Electronics|       P02|     Monitor|         1166704000|   3|
| Networking|       P05|      Router|          299992500|   1|
|     Office|       P04|     Printer|         1416678000|   1|
+-----------+----------+------------+-------------------+----+



In [ ]:
sales_with_rev = sales_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
window_spec_store = Window.partitionBy("store_id").orderBy("order_date", "order_id")

ans_14 = sales_with_rev.withColumn("running_gross_revenue", F.sum("gross_revenue").over(window_spec_store))
ans_14.show(10)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+---------------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|running_gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+---------------------+
|     180|2026-01-01|     S01|       P01|       5|     60000|           0|   Completed|           UPI|         EMP01|       300000|               300000|
|     360|2026-01-01|     S01|       P01|       1|     60000|           0|   Completed|           UPI|         EMP01|        60000|               360000|
|     540|2026-01-01|     S01|       P01|       5|     60000|           0|   Completed|           UPI|         EMP01|       300000|               660000|
|     720|2026-01-01|     S01|       P01|       1|     60000|           0|  

In [ ]:
ans_15 = sales_df.filter(F.col("order_status") == "Completed") \
                .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
                .withColumn("month", F.month("order_date")) \
                .groupBy("month") \
                .agg(F.sum("gross_revenue").alias("monthly_revenue")) \
                .orderBy("month")
ans_15.show()

+-----+---------------+
|month|monthly_revenue|
+-----+---------------+
|    1|     1278728500|
|    2|     1027585500|
|    3|     1212087500|
|    4|     1218921000|
|    5|     1277560500|
|    6|     1089027500|
+-----+---------------+



In [ ]:
df_filtered = sales_df.filter(F.col("order_status") == "Completed")
print("Transformation 1: Filter applied.")

df_with_rev = df_filtered.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
print("Transformation 2: Column added.")

df_selected = df_with_rev.select("order_id", "gross_revenue")
print("Transformation 3: Columns selected.")

print("Triggering action with show()...")
df_selected.show(5)

Transformation 1: Filter applied.
Transformation 2: Column added.
Transformation 3: Columns selected.
Triggering action with show()...
+--------+-------------+
|order_id|gross_revenue|
+--------+-------------+
|       3|        68000|
|       2|        10500|
|       4|        22500|
|       5|       144000|
|       1|        28000|
+--------+-------------+
only showing top 5 rows


In [ ]:
agg_df = sales_df.groupBy("product_id").agg(F.sum("quantity").alias("total_qty"))

print("=== Execution Plan ===")
agg_df.explain()

print("\n=== Executing Action ===")
result = agg_df.collect()

=== Execution Plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#33], functions=[sum(quantity#35)])
   +- HashAggregate(keys=[product_id#33], functions=[partial_sum(quantity#35)])
      +- Project [product_id#33, quantity#35]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=4357]
            :     +- Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, cast(((id#30L % 8) + 1) as int) AS quantity#35]
            :        +- Range (1, 100001, step=1, splits=2)
            +- Sort [product_id#0 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(product_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=4358]
                  +- Project [product_id#0]
                     +- Filter isnotnull(product_id#0)
                    

In [ ]:
cached_sales = sales_df.filter(F.col("order_status") == "Completed").cache()

start_time = time.time()
count1 = cached_sales.count()
print(f"First count: {count1}, Time taken: {time.time() - start_time:.4f} seconds")

start_time = time.time()
count2 = cached_sales.count()
print(f"Second count: {count2}, Time taken: {time.time() - start_time:.4f} seconds")

First count: 77923, Time taken: 5.0631 seconds
Second count: 77923, Time taken: 1.8748 seconds


In [ ]:
persisted_df = sales_df.filter(F.col("quantity") > 5).persist(StorageLevel.MEMORY_AND_DISK)

print("First Action Count:", persisted_df.count())
persisted_df.show(5)

persisted_df.unpersist()
print("Data unpersisted successfully.")

First Action Count: 37500
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|    Returned|           UPI|         EMP10|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      63|2026-03-05|     S02|       P04|       8|     17000|           0|    Returned|           UPI|         EMP04|
+--------+----------+--------+

In [ ]:
cached_agg = sales_df.groupBy("product_id").agg(F.count("order_id").alias("order_count")).cache()
cached_agg.count()

cached_agg.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [product_id#33, order_count#1708L]
      +- InMemoryRelation [product_id#33, order_count#1708L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=true
               +- == Final Plan ==
                  ResultQueryStage 3
                  +- *(3) HashAggregate(keys=[product_id#33], functions=[count(1)])
                     +- *(3) HashAggregate(keys=[product_id#33], functions=[partial_count(1)])
                        +- *(3) Project [product_id#33]
                           +- *(3) BroadcastHashJoin [product_id#33], [product_id#0], LeftOuter, BuildRight, false
                              :- ShuffleQueryStage 0
                              :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=4934]
                              :     +- *(1) Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33]


In [ ]:
print("Initial partition count:", sales_df.rdd.getNumPartitions())

repartitioned_df = sales_df.repartition(8)
print("New partition count:", repartitioned_df.rdd.getNumPartitions())

Initial partition count: 2
New partition count: 8


In [ ]:
repartitioned_df.withColumn("partition_id", F.spark_partition_id()) \
                .groupBy("partition_id") \
                .count() \
                .orderBy("partition_id") \
                .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|12500|
|           1|12500|
|           2|12500|
|           3|12500|
|           4|12500|
|           5|12500|
|           6|12500|
|           7|12500|
+------------+-----+



In [ ]:
store_repart_df = sales_df.repartition(6, "store_id")

store_repart_df.withColumn("partition_id", F.spark_partition_id()) \
               .select("partition_id", "store_id").distinct() \
               .orderBy("partition_id", "store_id") \
               .show()

+------------+--------+
|partition_id|store_id|
+------------+--------+
|           1|     S03|
|           3|     S02|
|           4|     S01|
|           5|     S04|
+------------+--------+



In [ ]:
df_12 = sales_df.repartition(12)

df_12.coalesce(4).explain()

df_12.repartition(4).explain()

=== Coalesce (4) Plan (No Shuffle) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 4
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=5977]
      +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=5971]
            :     +- Project [id#30L AS order_id#31L, date_add(2026-01-01, cast((id#30L % 180) as int)) AS order_date#32, concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, concat(S, lpad(cast(((FLOOR((cast(id#30L as double) / 3.0)) % 4) + 1) as string), 2, 0)) AS store_id#34, cast(((id#30L % 8) + 1) as int) AS quantity#35, CASE WHEN ((id#30L % 19) = 0) THEN null ELSE cast(((FL

In [ ]:
sales_df.repartition(4) \
        .write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv("sales_output_csv")

In [ ]:
# Steps 1 to 7
pipeline_df = (
    sales_df.filter(F.col("order_status") == "Completed")
    .fillna(0, subset=["discount_pct"])
    .join(products_df, "product_id", "inner")
    .join(stores_df, "store_id", "inner")
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .withColumn("discounted_revenue", F.col("gross_revenue") * (1 - (F.col("discount_pct") / 100)))
    .groupBy("region", "category")
    .agg(
        F.sum("discounted_revenue").alias("total_revenue"),
        F.count("order_id").alias("transaction_count")
    )
    .withColumn(
        "rank",
        F.dense_rank().over(Window.partitionBy("region").orderBy(F.col("total_revenue").desc()))
    )
)

pipeline_df.cache()

pipeline_df.show(truncate=False)

pipeline_df.repartition(2) \
           .write \
           .mode("overwrite") \
           .parquet("pipeline_output_parquet")

+------+-----------+-------------+-----------------+----+
|region|category   |total_revenue|transaction_count|rank|
+------+-----------+-------------+-----------------+----+
|South |Electronics|2.0242794E9  |19484            |1   |
|South |Office     |6.060058E8   |6495             |2   |
|South |Accessories|1.05018025E8 |6494             |3   |
|South |Networking |8.2684125E7  |6494             |4   |
|West  |Electronics|2.2992404E9  |12985            |1   |
|West  |Accessories|6.42985E7    |6493             |2   |
|North |Electronics|8.551824E8   |6492             |1   |
|North |Office     |4.12114E8    |6492             |2   |
|North |Networking |1.35026775E8 |6494             |3   |
+------+-----------+-------------+-----------------+----+



Practice set 2

In [ ]:
from google.colab import files
import zipfile
import os

# 1. Prompt to upload your zip file (e.g., dataset.zip)
uploaded = files.upload()

# 2. Extract all contents
for file_name in uploaded.keys():
    if file_name.endswith('.zip'):
        with zipfile.ZipFile(file_name, 'r') as zip_ref:
            zip_ref.extractall('./')  # Extracts to current working directory
        print(f'Successfully extracted: {file_name}')

Saving pyspark_logistics_practice_set_2.zip to pyspark_logistics_practice_set_2.zip
Successfully extracted: pyspark_logistics_practice_set_2.zip


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, countDistinct, sum as _sum, avg, when,
    to_date, datediff, date_format, dense_rank, row_number,
    spark_partition_id, explode
)
from pyspark.sql.window import Window
import time

spark = SparkSession.builder \
    .appName("LogisticsOperations") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

shipments_df = spark.read.csv("shipments_75000.csv", header=True, inferSchema=True)
extra_df = spark.read.csv("shipments_extra_12000.csv", header=True, inferSchema=True)
warehouse_df = spark.read.csv("warehouse_master.csv", header=True, inferSchema=True)
carrier_df = spark.read.csv("carrier_master.csv", header=True, inferSchema=True)
route_df = spark.read.csv("route_master.csv", header=True, inferSchema=True)
sensor_df = spark.read.json("warehouse_sensor_config.json", multiLine=True)

In [ ]:
shipments_df.printSchema()

print(f"Total Shipments: {shipments_df.count()}")

shipments_df.show(15, truncate=False)

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: string (nullable = true)
 |-- dispatch_date: string (nullable = true)
 |-- promised_delivery_date: string (nullable = true)
 |-- actual_delivery_date: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

Total Shipments: 75000
+-----------+------------+-------------+----

In [ ]:
selected_df = shipments_df.select("shipment_id", "warehouse_id", "service_level", "shipment_status", "shipping_fee_inr")
selected_df.show(10)

+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
| SHP0000001|         W08|      Express|      Delivered|          554.34|
| SHP0000002|         W08|     Standard|      Delivered|          414.56|
| SHP0000003|         W01|      Express|      Delivered|          724.64|
| SHP0000004|         W03|     Standard|      Delivered|          287.86|
| SHP0000005|         W03|      Economy|      Delivered|          251.68|
| SHP0000006|         W01|      Express|      Delivered|           477.9|
| SHP0000007|         W02|     Standard|     In Transit|          298.68|
| SHP0000008|         W03|      Economy|     In Transit|          284.52|
| SHP0000009|         W02|     Standard|      Delivered|          861.74|
| SHP0000010|         W02|      Economy|      Delivered|          309.76|
+-----------+------------+------------

In [ ]:
express_heavy_df = shipments_df.filter((col("service_level") == "Express") & (col("package_weight_kg") > 5))
express_heavy_df.show(10)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000003|  2026-01-31|   2026-02-01|            2026-02-03| 

In [ ]:
shipments_df.orderBy(col("shipping_fee_inr").desc()).show(20)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0044904|  2026-08-11|   2026-08-13|            2026-08-16| 

In [ ]:
shipments_df.groupBy("shipment_status").count().show()

+---------------+-----+
|shipment_status|count|
+---------------+-----+
|       Returned| 6130|
|           Lost| 2241|
|     In Transit|10472|
|      Cancelled| 3723|
|      Delivered|52434|
+---------------+-----+



In [ ]:
shipments_df.groupBy("service_level").agg(
    _sum("shipping_fee_inr").alias("total_revenue"),
    _sum("shipping_cost_inr").alias("total_cost"),
    (_sum("shipping_fee_inr") - _sum("shipping_cost_inr")).alias("total_profit"),
    ((_sum("shipping_fee_inr") - _sum("shipping_cost_inr")) / _sum("shipping_fee_inr") * 100).alias("profit_margin_pct")
).show()

+-------------+--------------------+--------------------+------------------+------------------+
|service_level|       total_revenue|          total_cost|      total_profit| profit_margin_pct|
+-------------+--------------------+--------------------+------------------+------------------+
|      Express|1.0058468059999986E7|   7350805.699999983| 2707662.360000002|26.919232072403737|
|     Standard|1.3437372619999977E7|1.0110465900000025E7|3326906.7199999522|24.758610288504137|
|      Economy|   4489918.680000002|   3386212.500000007|1103706.1799999946|24.581874609809063|
+-------------+--------------------+--------------------+------------------+------------------+



In [ ]:
shipments_df.join(warehouse_df, "warehouse_id") \
    .groupBy("region") \
    .agg(count("shipment_id").alias("shipment_count")) \
    .show()

+------+--------------+
|region|shipment_count|
+------+--------------+
| South|         33018|
|  East|          5306|
|  West|         26909|
| North|          9767|
+------+--------------+



In [ ]:
shipments_df.join(carrier_df, "carrier_id") \
    .join(route_df, "route_id") \
    .filter((col("shipping_cost_inr").isNotNull()) & (col("distance_km") > 0)) \
    .groupBy("carrier_name") \
    .agg(avg(col("shipping_cost_inr") / col("distance_km")).alias("avg_cost_per_km")) \
    .show()

+--------------------+------------------+
|        carrier_name|   avg_cost_per_km|
+--------------------+------------------+
|MetroRoute Logistics| 0.594275960971091|
|  AirSprint Couriers|0.6125755103088726|
|           CargoNest|0.5897283901696387|
|   BluePeak Delivery|0.6074832283099014|
|        ParcelBridge|0.5957409571811839|
|           RouteWise|0.5879324060720517|
|   SwiftLine Express|0.6032748764541479|
+--------------------+------------------+



In [ ]:
target_cols = ["carrier_id", "package_weight_kg", "shipping_cost_inr", "customer_rating"]
null_counts = shipments_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in target_cols
])
null_counts.show()

+----------+-----------------+-----------------+---------------+
|carrier_id|package_weight_kg|shipping_cost_inr|customer_rating|
+----------+-----------------+-----------------+---------------+
|       871|             1129|             1503|          26255|
+----------+-----------------+-----------------+---------------+



In [ ]:
# DataFrame 1: Fill null handling_minutes with 0
df_filled = shipments_df.fillna({"handling_minutes": 0})

# DataFrame 2: Exclude missing carrier_id
df_carrier_clean = shipments_df.filter(col("carrier_id").isNotNull())

In [ ]:
weight_categorized_df = shipments_df.withColumn(
    "weight_category",
    when(col("package_weight_kg") < 2, "Light")
    .when((col("package_weight_kg") >= 2) & (col("package_weight_kg") <= 10), "Medium")
    .when(col("package_weight_kg") > 10, "Heavy")
    .otherwise("Unknown")
)
weight_categorized_df.select("package_weight_kg", "weight_category").show(10)

+-----------------+---------------+
|package_weight_kg|weight_category|
+-----------------+---------------+
|             3.93|         Medium|
|             5.62|         Medium|
|             8.78|         Medium|
|             0.97|          Light|
|             NULL|        Unknown|
|             6.05|         Medium|
|             1.36|          Light|
|             2.04|         Medium|
|            18.73|          Heavy|
|             2.02|         Medium|
+-----------------+---------------+
only showing top 10 rows


In [ ]:
df_dates = shipments_df \
    .withColumn("booking_date_parsed", to_date(col("booking_date"), "yyyy-MM-dd")) \
    .withColumn("actual_delivery_date_parsed", to_date(col("actual_delivery_date"), "yyyy-MM-dd")) \
    .withColumn("promised_delivery_date_parsed", to_date(col("promised_delivery_date"), "yyyy-MM-dd")) \
    .withColumn("delivery_duration_days", datediff(col("actual_delivery_date_parsed"), col("booking_date_parsed")))

df_dates.filter(col("shipment_status") == "Delivered") \
    .select("shipment_id", "booking_date_parsed", "actual_delivery_date_parsed", "delivery_duration_days") \
    .show(10)

+-----------+-------------------+---------------------------+----------------------+
|shipment_id|booking_date_parsed|actual_delivery_date_parsed|delivery_duration_days|
+-----------+-------------------+---------------------------+----------------------+
| SHP0000001|         2026-05-24|                 2026-05-27|                     3|
| SHP0000002|         2026-05-17|                 2026-05-21|                     4|
| SHP0000003|         2026-01-31|                 2026-02-04|                     4|
| SHP0000004|         2026-03-12|                 2026-03-18|                     6|
| SHP0000005|         2026-08-16|                 2026-08-29|                    13|
| SHP0000006|         2026-02-06|                 2026-02-09|                     3|
| SHP0000009|         2026-08-12|                 2026-08-21|                     9|
| SHP0000010|         2026-06-01|                 2026-06-11|                    10|
| SHP0000011|         2026-04-24|                 2026-05-01|    

In [ ]:
df_performance = df_dates.withColumn(
    "delivery_performance",
    when(col("actual_delivery_date_parsed").isNull(), "Not Delivered")
    .when(col("actual_delivery_date_parsed") < col("promised_delivery_date_parsed"), "Early")
    .when(col("actual_delivery_date_parsed") == col("promised_delivery_date_parsed"), "On Time")
    .otherwise("Late")
)
df_performance.groupBy("delivery_performance").count().show()

+--------------------+-----+
|delivery_performance|count|
+--------------------+-----+
|             On Time|22049|
|                Late|21900|
|               Early| 8485|
|       Not Delivered|22566|
+--------------------+-----+



In [ ]:
df_performance.filter(col("shipment_status") == "Delivered") \
    .withColumn("booking_month", date_format(col("booking_date_parsed"), "yyyy-MM")) \
    .groupBy("booking_month") \
    .agg(
        count("shipment_id").alias("delivered_count"),
        _sum("shipping_fee_inr").alias("total_fees_inr")
    ) \
    .orderBy("booking_month") \
    .show()

+-------------+---------------+------------------+
|booking_month|delivered_count|    total_fees_inr|
+-------------+---------------+------------------+
|      2026-01|           6710| 2525138.440000005|
|      2026-02|           6097|2268113.5000000037|
|      2026-03|           6810|2530456.2400000026|
|      2026-04|           6379|2389163.4600000014|
|      2026-05|           6858|2549683.9999999963|
|      2026-06|           6615|2447590.1200000006|
|      2026-07|           6773|2512355.2199999997|
|      2026-08|           6192|2306432.5199999996|
+-------------+---------------+------------------+



In [ ]:
carrier_rev = shipments_df.join(warehouse_df, "warehouse_id") \
    .groupBy("region", "carrier_id") \
    .agg(_sum("shipping_fee_inr").alias("total_fee"))

window_region = Window.partitionBy("region").orderBy(col("total_fee").desc())

carrier_rev.withColumn("rank", dense_rank().over(window_region)).show(20)

+------+----------+------------------+----+
|region|carrier_id|         total_fee|rank|
+------+----------+------------------+----+
|  East|       C01|393862.61999999994|   1|
|  East|       C02|         373590.02|   2|
|  East|       C03|302260.92000000004|   3|
|  East|       C05|243322.91999999998|   4|
|  East|       C07|222025.85999999993|   5|
|  East|       C04|211583.64000000004|   6|
|  East|       C06| 195810.3399999999|   7|
|  East|      NULL|           24594.6|   8|
| North|       C02| 731442.0999999994|   1|
| North|       C01| 653223.6200000005|   2|
| North|       C03| 597883.4600000005|   3|
| North|       C05|464601.85999999975|   4|
| North|       C07|425396.69999999995|   5|
| North|       C04|393181.07999999984|   6|
| North|       C06|338296.45999999985|   7|
| North|      NULL|45235.700000000004|   8|
| South|       C02|2442652.5400000014|   1|
| South|       C01| 2271284.540000001|   2|
| South|       C03|1929008.1199999978|   3|
| South|       C07|1519441.49999

In [ ]:
window_wh_latest = Window.partitionBy("warehouse_id").orderBy(col("booking_date").desc(), col("shipment_id").desc())

shipments_df.withColumn("rn", row_number().over(window_wh_latest)) \
    .filter(col("rn") == 1).drop("rn") \
    .show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0072346|  2026-08-28|         NULL|            2026-09-03| 

In [ ]:
window_running = Window.partitionBy("warehouse_id").orderBy("booking_date", "shipment_id")

shipments_df.withColumn("running_total_fees", _sum("shipping_fee_inr").over(window_running)) \
    .select("warehouse_id", "booking_date", "shipment_id", "shipping_fee_inr", "running_total_fees") \
    .show(20)

+------------+------------+-----------+----------------+------------------+
|warehouse_id|booking_date|shipment_id|shipping_fee_inr|running_total_fees|
+------------+------------+-----------+----------------+------------------+
|         W01|  2026-01-01| SHP0004062|          312.48|            312.48|
|         W01|  2026-01-01| SHP0006739|          227.02|             539.5|
|         W01|  2026-01-01| SHP0007553|          316.02|            855.52|
|         W01|  2026-01-01| SHP0008505|          572.68|1428.1999999999998|
|         W01|  2026-01-01| SHP0009674|           322.9|            1751.1|
|         W01|  2026-01-01| SHP0011312|           390.1|            2141.2|
|         W01|  2026-01-01| SHP0011588|          421.72|           2562.92|
|         W01|  2026-01-01| SHP0011698|          190.22|           2753.14|
|         W01|  2026-01-01| SHP0011973|          384.64|3137.7799999999997|
|         W01|  2026-01-01| SHP0017305|          203.48|3341.2599999999998|
|         W0

In [ ]:
combined_df = shipments_df.unionByName(extra_df)
print(f"Combined total count: {combined_df.count()}")

Combined total count: 87000


In [ ]:
print(f"Rows before deduplication: {combined_df.count()}")

# Find duplicates
duplicate_ids = combined_df.groupBy("shipment_id").count().filter(col("count") > 1)
print(f"Number of repeated shipment IDs: {duplicate_ids.count()}")

# Deduplicate
dedup_df = combined_df.dropDuplicates(["shipment_id"])
print(f"Rows after deduplication: {dedup_df.count()}")

Rows before deduplication: 87000
Number of repeated shipment IDs: 750
Rows after deduplication: 86250


In [ ]:
exploded_sensors = sensor_df.select(
    col("warehouse_id"),
    col("environment.temperature_range_c.minimum").alias("min_temp_c"),
    col("environment.temperature_range_c.maximum").alias("max_temp_c"),
    explode(col("sensors")).alias("sensor")
).select(
    "warehouse_id",
    "min_temp_c",
    "max_temp_c",
    col("sensor.sensor_id"),
    col("sensor.type").alias("sensor_type"),
    col("sensor.enabled").alias("sensor_enabled")
)

exploded_sensors.show(20, truncate=False)

+------------+----------+----------+---------+-----------+--------------+
|warehouse_id|min_temp_c|max_temp_c|sensor_id|sensor_type|sensor_enabled|
+------------+----------+----------+---------+-----------+--------------+
|W01         |18        |32        |W01-S1   |Temperature|false         |
|W01         |18        |32        |W01-S2   |Humidity   |true          |
|W01         |18        |32        |W01-S3   |Door       |true          |
|W02         |18        |32        |W02-S1   |Temperature|true          |
|W02         |18        |32        |W02-S2   |Humidity   |true          |
|W02         |18        |32        |W02-S3   |Door       |true          |
|W03         |18        |32        |W03-S1   |Temperature|true          |
|W03         |18        |32        |W03-S2   |Humidity   |true          |
|W03         |18        |32        |W03-S3   |Door       |true          |
|W04         |18        |32        |W04-S1   |Temperature|true          |
|W04         |18        |32        |W0

In [ ]:
pipeline_df = shipments_df \
    .filter(col("shipment_status") == "Delivered") \
    .join(warehouse_df, "warehouse_id") \
    .withColumn("gross_profit", col("shipping_fee_inr") - col("shipping_cost_inr")) \
    .groupBy("region") \
    .agg(avg("gross_profit").alias("avg_profit")) \
    .orderBy(col("avg_profit").desc())

pipeline_df.explain(True)
pipeline_df.show()

== Parsed Logical Plan ==
'Sort ['avg_profit DESC NULLS LAST], true
+- Aggregate [region#94], [region#94, avg(gross_profit#1142) AS avg_profit#1143]
   +- Project [warehouse_id#22, shipment_id#17, booking_date#18, dispatch_date#19, promised_delivery_date#20, actual_delivery_date#21, carrier_id#23, route_id#24, customer_segment#25, service_level#26, package_weight_kg#27, shipping_fee_inr#28, shipping_cost_inr#29, shipment_status#30, delivery_attempts#31, damage_flag#32, customer_rating#33, handling_minutes#34, payment_mode#35, priority_flag#36, warehouse_name#92, city#93, region#94, capacity_units#95, storage_type#96, ... 1 more fields]
      +- Project [warehouse_id#22, shipment_id#17, booking_date#18, dispatch_date#19, promised_delivery_date#20, actual_delivery_date#21, carrier_id#23, route_id#24, customer_segment#25, service_level#26, package_weight_kg#27, shipping_fee_inr#28, shipping_cost_inr#29, shipment_status#30, delivery_attempts#31, damage_flag#32, customer_rating#33, handling

In [ ]:
delivered_df = shipments_df.filter(col("shipment_status") == "Delivered")

# Without cache
start_time = time.time()
c1 = delivered_df.count()
a1 = delivered_df.groupBy("service_level").agg(avg("shipping_fee_inr")).collect()
print(f"Execution time WITHOUT cache: {time.time() - start_time:.4f} seconds")

# With cache
delivered_df.cache()
delivered_df.count()

start_time = time.time()
c2 = delivered_df.count()
a2 = delivered_df.groupBy("service_level").agg(avg("shipping_fee_inr")).collect()
print(f"Execution time WITH cache: {time.time() - start_time:.4f} seconds")

delivered_df.unpersist()

Execution time WITHOUT cache: 2.2128 seconds
Execution time WITH cache: 0.5138 seconds


DataFrame[shipment_id: string, booking_date: string, dispatch_date: string, promised_delivery_date: string, actual_delivery_date: string, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

In [ ]:
from pyspark.storagelevel import StorageLevel

express_df = shipments_df.filter(col("service_level") == "Express")
express_df.persist(StorageLevel.MEMORY_AND_DISK)

print(f"Express Count: {express_df.count()}")
express_df.groupBy("shipment_status").count().show()

print(f"Is persisted: {express_df.is_cached}")

express_df.unpersist()

Express Count: 21090
+---------------+-----+
|shipment_status|count|
+---------------+-----+
|       Returned| 1767|
|           Lost|  575|
|     In Transit| 2944|
|      Cancelled|  985|
|      Delivered|14819|
+---------------+-----+

Is persisted: True


DataFrame[shipment_id: string, booking_date: string, dispatch_date: string, promised_delivery_date: string, actual_delivery_date: string, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

In [ ]:
df_repart8 = shipments_df.repartition(8)
df_repart8.withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 9375|
|           1| 9374|
|           2| 9376|
|           3| 9375|
|           4| 9375|
|           5| 9375|
|           6| 9375|
|           7| 9375|
+------------+-----+



In [ ]:
df_12 = shipments_df.repartition(12)

df_coalesced = df_12.coalesce(3)
df_repart3 = df_12.repartition(3)

print("--- Coalesce (3) Plan ---")
df_coalesced.explain()

print("--- Repartition (3) Plan ---")
df_repart3.explain()

--- Coalesce (3) Plan ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=2229]
      +- FileScan csv [shipment_id#17,booking_date#18,dispatch_date#19,promised_delivery_date#20,actual_delivery_date#21,warehouse_id#22,carrier_id#23,route_id#24,customer_segment#25,service_level#26,package_weight_kg#27,shipping_fee_inr#28,shipping_cost_inr#29,shipment_status#30,delivery_attempts#31,damage_flag#32,customer_rating#33,handling_minutes#34,payment_mode#35,priority_flag#36] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/shipments_75000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<shipment_id:string,booking_date:string,dispatch_date:string,promised_delivery_date:string,...


--- Repartition (3) Plan ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=2238]
  

In [ ]:
joined_export_df = shipments_df.join(warehouse_df, "warehouse_id")

joined_export_df.repartition(4) \
    .write \
    .mode("overwrite") \
    .parquet("output_shipments_parquet")

import os
print("Generated Parquet Files:", os.listdir("output_shipments_parquet"))

Generated Parquet Files: ['._SUCCESS.crc', '.part-00001-1ae0fc14-ad61-48b5-b5fb-810578598faf-c000.snappy.parquet.crc', 'part-00000-1ae0fc14-ad61-48b5-b5fb-810578598faf-c000.snappy.parquet', '_SUCCESS', '.part-00002-1ae0fc14-ad61-48b5-b5fb-810578598faf-c000.snappy.parquet.crc', 'part-00002-1ae0fc14-ad61-48b5-b5fb-810578598faf-c000.snappy.parquet', 'part-00003-1ae0fc14-ad61-48b5-b5fb-810578598faf-c000.snappy.parquet', '.part-00000-1ae0fc14-ad61-48b5-b5fb-810578598faf-c000.snappy.parquet.crc', '.part-00003-1ae0fc14-ad61-48b5-b5fb-810578598faf-c000.snappy.parquet.crc', 'part-00001-1ae0fc14-ad61-48b5-b5fb-810578598faf-c000.snappy.parquet']


In [ ]:
raw_shipments = shipments_df.unionByName(extra_df).dropDuplicates(["shipment_id"])

avg_metrics = raw_shipments.groupBy("service_level").agg(
    avg("package_weight_kg").alias("avg_weight"),
    avg("shipping_cost_inr").alias("avg_cost")
)

cleaned_shipments = raw_shipments.join(avg_metrics, "service_level") \
    .withColumn("package_weight_kg", when(col("package_weight_kg").isNull(), col("avg_weight")).otherwise(col("package_weight_kg"))) \
    .withColumn("shipping_cost_inr", when(col("shipping_cost_inr").isNull(), col("avg_cost")).otherwise(col("shipping_cost_inr"))) \
    .drop("avg_weight", "avg_cost")

full_pipeline_df = cleaned_shipments \
    .join(warehouse_df, "warehouse_id") \
    .join(carrier_df, "carrier_id") \
    .join(route_df, "route_id") \
    .withColumn("booking_dt", to_date(col("booking_date"), "yyyy-MM-dd")) \
    .withColumn("actual_dt", to_date(col("actual_delivery_date"), "yyyy-MM-dd")) \
    .withColumn("promised_dt", to_date(col("promised_delivery_date"), "yyyy-MM-dd")) \
    .withColumn("delivery_duration", datediff(col("actual_dt"), col("booking_dt"))) \
    .withColumn("is_late", when(col("actual_dt") > col("promised_dt"), 1).otherwise(0)) \
    .withColumn("profit_inr", col("shipping_fee_inr") - col("shipping_cost_inr"))

agg_performance = full_pipeline_df.groupBy("region", "carrier_name").agg(
    _sum("shipping_fee_inr").alias("total_revenue"),
    _sum("profit_inr").alias("total_profit"),
    (avg("is_late") * 100).alias("late_delivery_pct"),
    ((1 - avg("is_late")) * 100).alias("ontime_delivery_pct")
)

# 5. Rank carriers within region by on-time delivery %
rank_window = Window.partitionBy("region").orderBy(col("ontime_delivery_pct").desc())
final_report_df = agg_performance.withColumn("carrier_rank", dense_rank().over(rank_window))

# 6. Cache and perform actions
final_report_df.cache()

print("--- Regional Carrier Performance Summary ---")
final_report_df.show(50, truncate=False)

# 7. Write report to partitioned Parquet files
final_report_df.write \
    .partitionBy("region") \
    .mode("overwrite") \
    .parquet("final_logistics_performance_report")

final_report_df.unpersist()

--- Regional Carrier Performance Summary ---
+------+--------------------+------------------+------------------+------------------+-------------------+------------+
|region|carrier_name        |total_revenue     |total_profit      |late_delivery_pct |ontime_delivery_pct|carrier_rank|
+------+--------------------+------------------+------------------+------------------+-------------------+------------+
|South |AirSprint Couriers  |1551724.859999998 |371924.1184938841 |28.16386247256767 |71.83613752743233  |1           |
|South |SwiftLine Express   |2632393.9999999977|625897.879253923  |28.24014795845782 |71.75985204154219  |2           |
|South |CargoNest           |1661626.059999999 |398333.192601794  |28.45216606498195 |71.54783393501805  |3           |
|South |RouteWise           |1403377.4599999995|333180.4391173842 |29.65407974650119 |70.34592025349882  |4           |
|South |ParcelBridge        |2799226.2399999974|687408.2621074994 |29.697050225883604|70.3029497741164   |5        

DataFrame[region: string, carrier_name: string, total_revenue: double, total_profit: double, late_delivery_pct: double, ontime_delivery_pct: double, carrier_rank: int]